# 05.2 – PostgreSQL database and SQL queries from Python

**Owner:** Luca · **Extra points:** PostgreSQL (not SQLite) with joins and aggregations from Python

This notebook repeats the work of `05_sqlite_database.ipynb` in **PostgreSQL**, a database server. SQLite is a single file; PostgreSQL runs as a separate program and Python connects to it with a user name and password. The notebook

1. connects to the PostgreSQL server with SQLAlchemy and psycopg2,
2. creates the tables (PostgreSQL data types such as `BOOLEAN`, `DATE`, `DOUBLE PRECISION`),
3. loads the same CSV files,
4. runs the same project questions with `JOIN`, `GROUP BY`, `HAVING` and `WITH`, adapted to the PostgreSQL syntax,
5. adds two things SQLite cannot do as simply: a **window function** (`RANK() OVER`) and a **median** (`percentile_cont`),
6. checks the results against pandas and against the SQLite database.

**Before running:** PostgreSQL must be installed and started. In the Codespace run `bash src/setup_postgres_codespace.sh` once, and `sudo service postgresql start` after each restart of the Codespace.

Syntax differences to SQLite that matter here: `GROUP_CONCAT` becomes `STRING_AGG`; `SUM(condition)` becomes `COUNT(*) FILTER (WHERE condition)`; `ROUND(x, n)` needs a `numeric` value, so `AVG(...)::numeric` is used; there is no `AUTOINCREMENT`.

In [1]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# Find the repository root (the folder with requirements.txt)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
SQLITE_PATH = ROOT / "db" / "sci_alpi.db"

# Connection settings. The defaults match src/setup_postgres_codespace.sh (local development only).
PG_HOST = os.getenv("PGHOST", "localhost")
PG_PORT = os.getenv("PGPORT", "5432")
PG_DB = os.getenv("PGDATABASE", "sci_alpi")
PG_USER = os.getenv("PGUSER", "sci")
PG_PASSWORD = os.getenv("PGPASSWORD", "sci_alpi")

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    connect_args={"options": "-c client_encoding=utf8"},   # resort names contain characters such as the long dash
)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def sql(query, params=None):
    """Run a SELECT query on PostgreSQL and return the result as a pandas table."""
    return pd.read_sql_query(text(query), engine, params=params, coerce_float=True)

try:
    with engine.connect() as con:
        version = con.execute(text("SELECT version()")).scalar()
    print("Connected:", version.split(",")[0])
except Exception as err:
    raise SystemExit(
        "Cannot connect to PostgreSQL. Run `bash src/setup_postgres_codespace.sh` once and "
        "`sudo service postgresql start`, then run this notebook again.\n" + str(err)[:300]
    )

Connected: PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu


## 1. Schema

The tables are the same as in the SQLite notebook. In PostgreSQL the flags are real `BOOLEAN` columns and the exchange-rate date is a real `DATE`. `DROP TABLE IF EXISTS ... CASCADE` makes the notebook repeatable.

In [2]:
SCHEMA = '''
DROP VIEW  IF EXISTS v_latest_earnings;
DROP TABLE IF EXISTS snow_winters_pilot, snow_summary_pilot, station_winters, fx_eur_chf_daily,
                     country_stats, closures, resorts, resort_roster CASCADE;

CREATE TABLE resort_roster (
    resort_id                        TEXT PRIMARY KEY,
    resort_name                      TEXT NOT NULL,
    url                              TEXT NOT NULL,
    country_memberships              TEXT,
    country_for_analysis             TEXT NOT NULL CHECK (country_for_analysis IN ('AT','CH','FR','IT','cross_border')),
    source_row_count                 INTEGER,
    elevation_base_m                 INTEGER,
    elevation_top_m                  INTEGER,
    latitude                         DOUBLE PRECISION,
    longitude                        DOUBLE PRECISION,
    scope_decision                   TEXT,
    include_in_alpine_candidate_set  BOOLEAN NOT NULL,
    base_elevation_at_point_verified BOOLEAN,
    snow_method_validated            BOOLEAN
);

CREATE TABLE resorts (
    row_id               INTEGER PRIMARY KEY,
    resort_id            TEXT NOT NULL REFERENCES resort_roster(resort_id),
    resort_name          TEXT NOT NULL,
    country              TEXT NOT NULL CHECK (country IN ('AT','CH','FR','IT')),
    elevation_base_m     INTEGER,
    elevation_top_m      INTEGER,
    slope_km             DOUBLE PRECISION,
    lifts_count          DOUBLE PRECISION,
    price_day_raw        DOUBLE PRECISION,
    currency             TEXT CHECK (currency IN ('CHF','EUR')),
    rating_stars         DOUBLE PRECISION,
    snow_cannons         INTEGER,
    price_day_eur        DOUBLE PRECISION,
    latitude             DOUBLE PRECISION,
    longitude            DOUBLE PRECISION,
    dist_zurich_km       DOUBLE PRECISION,
    dist_geneva_km       DOUBLE PRECISION,
    dist_milan_km        DOUBLE PRECISION,
    dist_turin_km        DOUBLE PRECISION,
    dist_munich_km       DOUBLE PRECISION,
    dist_lyon_km         DOUBLE PRECISION,
    dist_nearest_city_km DOUBLE PRECISION,
    hotels_4_5_star_10km INTEGER
);

CREATE TABLE closures (
    closure_id    INTEGER PRIMARY KEY,
    facility_name TEXT NOT NULL,
    comune        TEXT,
    province      TEXT,
    region        TEXT NOT NULL,
    status        TEXT NOT NULL CHECK (status IN ('dismesso','chiuso temporaneamente')),
    altitude_text TEXT,
    facility_type TEXT,
    year_built    TEXT,
    year_closed   TEXT,
    source_page   INTEGER
);

CREATE TABLE country_stats (
    country               TEXT NOT NULL CHECK (country IN ('AT','CH','FR','IT')),
    year                  INTEGER NOT NULL,
    pli_eu27              DOUBLE PRECISION,
    net_earnings_year_eur DOUBLE PRECISION,
    net_earnings_hour_eur DOUBLE PRECISION,
    PRIMARY KEY (country, year)
);

CREATE TABLE fx_eur_chf_daily (
    date    DATE PRIMARY KEY,
    eur_chf DOUBLE PRECISION NOT NULL      -- CHF per 1 EUR
);

CREATE TABLE station_winters (
    station_id     TEXT NOT NULL,
    station_name   TEXT,
    lat            DOUBLE PRECISION,
    lon            DOUBLE PRECISION,
    elev_m         DOUBLE PRECISION,
    winter         INTEGER NOT NULL,
    period         TEXT NOT NULL CHECK (period IN ('past','recent')),
    snow_days_30cm INTEGER,
    observed_days  INTEGER,
    source         TEXT,
    PRIMARY KEY (station_id, winter)
);

CREATE TABLE snow_summary_pilot (
    resort_id      TEXT PRIMARY KEY,
    name           TEXT,
    country        TEXT,
    elev_base_m    INTEGER,
    days_past      DOUBLE PRECISION,
    days_recent    DOUBLE PRECISION,
    lost_snow_days DOUBLE PRECISION,
    risk_class     TEXT
);

CREATE TABLE snow_winters_pilot (
    resort_id      TEXT NOT NULL,
    winter         INTEGER NOT NULL,
    period         TEXT,
    snow_days_30cm INTEGER,
    elev_query_m   INTEGER,
    PRIMARY KEY (resort_id, winter)
);

-- An index speeds up searches and joins on a column
CREATE INDEX idx_resorts_resort_id ON resorts (resort_id);
CREATE INDEX idx_resorts_country   ON resorts (country);

-- A view is a saved query: the latest year with net hourly earnings for each country
CREATE VIEW v_latest_earnings AS
SELECT cs.country, cs.year, cs.net_earnings_hour_eur
FROM country_stats cs
JOIN (SELECT country, MAX(year) AS year
      FROM country_stats
      WHERE net_earnings_hour_eur IS NOT NULL
      GROUP BY country) m
  ON m.country = cs.country AND m.year = cs.year;
'''
with engine.begin() as con:           # begin() = one transaction: everything is saved, or nothing
    con.exec_driver_sql(SCHEMA)

sql("""
    SELECT table_type, table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_type DESC, table_name
""")

,table_type,table_name
0,VIEW,v_latest_earnings
1,BASE TABLE,closures
2,BASE TABLE,country_stats
3,BASE TABLE,fx_eur_chf_daily
4,BASE TABLE,resort_roster
5,BASE TABLE,resorts
6,BASE TABLE,snow_summary_pilot
7,BASE TABLE,snow_winters_pilot
8,BASE TABLE,station_winters


## 2. Load the CSV files

Same helper as in the SQLite notebook: it appends the CSV file to the table and compares the row counts.

In [3]:
log = []

def load(table, df, source):
    df.to_sql(table, engine, if_exists="append", index=False, method="multi", chunksize=1000)
    n_db = sql(f"SELECT COUNT(*) AS n FROM {table}")["n"].iloc[0]
    log.append({"table": table, "source": source, "rows_in_file": len(df), "rows_in_table": int(n_db), "lost": len(df) - int(n_db)})

def table_columns(table):
    return list(sql("SELECT column_name FROM information_schema.columns WHERE table_name = :t ORDER BY ordinal_position",
                    {"t": table})["column_name"])

# 1) Resort roster
roster = pd.read_csv(PROCESSED / "resorts_snow_roster.csv")
for col in ["include_in_alpine_candidate_set", "base_elevation_at_point_verified", "snow_method_validated"]:
    roster[col] = roster[col].astype(bool)
load("resort_roster", roster[table_columns("resort_roster")], "resorts_snow_roster.csv")

# 2) Resorts (the resort_id is the last part of the Skiresort URL)
resorts = pd.read_csv(PROCESSED / "resorts_features.csv")
resorts.insert(0, "resort_id", resorts["url"].str.rstrip("/").str.split("/").str[-1])
resorts.insert(0, "row_id", range(1, len(resorts) + 1))
load("resorts", resorts[table_columns("resorts")], "resorts_features.csv")

# 3) Legambiente closures
closures = pd.read_csv(PROCESSED / "legambiente_closures.csv")
closures.insert(0, "closure_id", range(1, len(closures) + 1))
load("closures", closures, "legambiente_closures.csv")

# 4) Country statistics, exchange rate, station winters, pilot snow
load("country_stats", pd.read_csv(PROCESSED / "country_stats.csv"), "country_stats.csv")
fx = pd.read_csv(PROCESSED / "fx_eur_chf_daily.csv")
fx["date"] = pd.to_datetime(fx["date"]).dt.date
load("fx_eur_chf_daily", fx, "fx_eur_chf_daily.csv")
stations = pd.read_csv(PROCESSED / "meteoswiss_station_winters.csv")
load("station_winters", stations[table_columns("station_winters")], "meteoswiss_station_winters.csv")
load("snow_summary_pilot", pd.read_csv(PROCESSED / "snow_summary_10resorts.csv"), "snow_summary_10resorts.csv")
load("snow_winters_pilot", pd.read_csv(PROCESSED / "snow_winters_pilot.csv"), "snow_winters_pilot.csv")

load_log = pd.DataFrame(log)
load_log

,table,source,rows_in_file,rows_in_table,lost
0,resort_roster,resorts_snow_roster.csv,293,293,0
1,resorts,resorts_features.csv,300,300,0
2,closures,legambiente_closures.csv,184,184,0
3,country_stats,country_stats.csv,123,123,0
4,fx_eur_chf_daily,fx_eur_chf_daily.csv,4098,4098,0
5,station_winters,meteoswiss_station_winters.csv,400,400,0
6,snow_summary_pilot,snow_summary_10resorts.csv,10,10,0
7,snow_winters_pilot,snow_winters_pilot.csv,20,20,0


## 3. Quality checks with SQL

In PostgreSQL the foreign key is checked at every insert, so a wrong `resort_id` would have stopped the loading. The checks below show duplicates and missing values.

In [4]:
# Resort IDs that appear in more than one row (STRING_AGG replaces GROUP_CONCAT)
dups = sql('''
    SELECT resort_id,
           COUNT(*)                                        AS n_rows,
           STRING_AGG(country, ',' ORDER BY row_id)        AS countries,
           STRING_AGG(price_day_eur::text, ',' ORDER BY row_id) AS prices_eur
    FROM resorts
    GROUP BY resort_id
    HAVING COUNT(*) > 1
    ORDER BY resort_id
''')
dups

,resort_id,n_rows,countries,prices_eur
0,arosa-lenzerheide,2,"CH,CH","93.45,89"
1,espace-san-bernardo-la-rosiere-la-thuile,2,"IT,FR","59,59"
2,ischgl-samnaun-silvretta-arena,2,"CH,AT","87.15,83"
3,les-portes-du-soleil-morzine-avoriaz-les-gets-...,2,"CH,FR","76.65,73"
4,obertauern,2,"AT,AT","72,72"
5,via-lattea-sestriere-sauze-doulx-san-sicario-c...,2,"IT,FR","66,66"
6,zermatt-breuil-cervinia-valtournenche-matterhorn,2,"CH,IT","121.8,116"


In [5]:
# Missing and constant values (COUNT(*) FILTER (WHERE ...) replaces SUM(condition))
sql('''
    SELECT COUNT(*)                                          AS n_rows,
           COUNT(*) FILTER (WHERE lifts_count IS NULL)       AS lifts_count_missing,
           COUNT(*) FILTER (WHERE price_day_raw IS NULL)     AS price_raw_missing,
           COUNT(*) FILTER (WHERE price_day_eur IS NULL)     AS price_eur_missing,
           COUNT(*) FILTER (WHERE snow_cannons = 0)          AS snow_cannons_zero,
           COUNT(DISTINCT snow_cannons)                      AS snow_cannons_distinct_values
    FROM resorts
''')

,n_rows,lifts_count_missing,price_raw_missing,price_eur_missing,snow_cannons_zero,snow_cannons_distinct_values
0,300,300,2,0,300,1


## 4. Project questions in PostgreSQL

### 4.1 Price and elevation by country

In [6]:
q_country = sql('''
    SELECT country,
           COUNT(*)                                  AS n_rows,
           ROUND(AVG(price_day_eur)::numeric, 1)     AS avg_day_pass_eur,
           ROUND(AVG(elevation_base_m)::numeric, 0)  AS avg_base_m,
           ROUND(AVG(slope_km)::numeric, 1)          AS avg_slope_km,
           ROUND(AVG(rating_stars)::numeric, 2)      AS avg_rating
    FROM resorts
    GROUP BY country
    ORDER BY avg_day_pass_eur DESC
''')
q_country

,country,n_rows,avg_day_pass_eur,avg_base_m,avg_slope_km,avg_rating
0,CH,76,73.2,1282.0,49.7,3.72
1,AT,76,72.6,1066.0,31.3,4.15
2,IT,75,61.5,1320.0,48.3,3.75
3,FR,73,49.7,1336.0,100.8,3.58


### 4.2 Ski areas by analysis country (`JOIN` with the roster)

In [7]:
q_roster = sql('''
    SELECT rr.country_for_analysis                    AS country_group,
           COUNT(DISTINCT rr.resort_id)               AS n_ski_areas,
           COUNT(*)                                   AS n_price_rows,
           ROUND(AVG(r.price_day_eur)::numeric, 1)    AS avg_day_pass_eur
    FROM resorts r
    JOIN resort_roster rr ON rr.resort_id = r.resort_id
    WHERE rr.include_in_alpine_candidate_set          -- a BOOLEAN column can be used directly
    GROUP BY rr.country_for_analysis
    ORDER BY n_ski_areas DESC
''')
q_roster

,country_group,n_ski_areas,n_price_rows,avg_day_pass_eur
0,AT,74,75,72.4
1,CH,72,73,72.3
2,IT,69,69,61.2
3,FR,55,55,50.4
4,cross_border,5,10,80.8


### 4.3 Affordability: working hours needed for a day pass (SQ4)

Same question as in the SQLite notebook: `price_day_eur / net_earnings_hour_eur`, with the latest year of earnings for each country taken from the view.

In [8]:
q_hours = sql('''
    SELECT r.country,
           e.year                                                         AS earnings_year,
           COUNT(*)                                                       AS n_resorts,
           ROUND(AVG(r.price_day_eur)::numeric, 1)                        AS avg_day_pass_eur,
           ROUND(e.net_earnings_hour_eur::numeric, 1)                     AS net_hourly_earnings_eur,
           ROUND(AVG(r.price_day_eur / e.net_earnings_hour_eur)::numeric, 2) AS work_hours_per_day_pass
    FROM resorts r
    JOIN v_latest_earnings e ON e.country = r.country
    WHERE r.price_day_eur IS NOT NULL
    GROUP BY r.country, e.year, e.net_earnings_hour_eur
    ORDER BY work_hours_per_day_pass DESC
''')
q_hours

,country,earnings_year,n_resorts,avg_day_pass_eur,net_hourly_earnings_eur,work_hours_per_day_pass
0,IT,2025,75,61.5,15.8,3.90
1,AT,2025,76,72.6,24.1,3.02
2,FR,2025,73,49.7,18.4,2.70
3,CH,2023,76,73.2,46.6,1.57


**Reading the result.** The result is the same as in SQLite: about 3.9 hours of net work for a day pass in Italy, 3.0 in Austria, 2.7 in France and 1.6 in Switzerland. PostgreSQL needs the columns `e.year` and `e.net_earnings_hour_eur` in `GROUP BY`, because every selected column must be grouped or aggregated.

### 4.4 Exchange-rate check for the Swiss prices (`WITH`)

In [9]:
q_fx = sql('''
    WITH fx AS (
        SELECT AVG(eur_chf) AS avg_chf_per_eur
        FROM fx_eur_chf_daily
        WHERE date >= DATE '2025-01-01'
    )
    SELECT COUNT(*)                                                    AS n_chf_prices,
           ROUND(fx.avg_chf_per_eur::numeric, 4)                       AS avg_chf_per_eur_2025,
           ROUND((1.0 / fx.avg_chf_per_eur)::numeric, 4)               AS eur_per_chf_2025,
           ROUND(AVG(r.price_day_eur)::numeric, 2)                     AS avg_price_eur_fixed_1_05,
           ROUND(AVG(r.price_day_raw / fx.avg_chf_per_eur)::numeric, 2) AS avg_price_eur_2025_rate,
           ROUND((100.0 * (AVG(r.price_day_raw / fx.avg_chf_per_eur) / AVG(r.price_day_eur) - 1))::numeric, 1) AS difference_pct
    FROM resorts r CROSS JOIN fx
    WHERE r.currency = 'CHF' AND r.price_day_raw IS NOT NULL
    GROUP BY fx.avg_chf_per_eur
''')
q_fx

,n_chf_prices,avg_chf_per_eur_2025,eur_per_chf_2025,avg_price_eur_fixed_1_05,avg_price_eur_2025_rate,difference_pct
0,75,0.937,1.0672,72.98,74.18,1.6


### 4.5 Closed ski facilities by region

In [10]:
q_closures = sql('''
    SELECT region,
           COUNT(*) FILTER (WHERE status = 'dismesso')               AS dismissed,
           COUNT(*) FILTER (WHERE status = 'chiuso temporaneamente') AS temporarily_closed,
           COUNT(*)                                                  AS total
    FROM closures
    GROUP BY region
    ORDER BY total DESC, region
''')
q_closures

,region,dismissed,temporarily_closed,total
0,Piemonte,38,7,45
1,Lombardia,32,2,34
2,Veneto,16,3,19
3,Emilia-Romagna,12,6,18
4,Abruzzo,10,5,15
5,Friuli Venezia Giulia,7,1,8
6,Toscana,7,0,7
7,Valle d’Aosta,5,1,6
8,Lazio,2,3,5
9,Basilicata,2,2,4


### 4.6 Measured snow days: past vs recent by elevation band

In [11]:
q_snow = sql('''
    SELECT CASE WHEN elev_m < 1500 THEN 'below 1500 m' ELSE '1500 m and above' END AS elevation_band,
           period,
           COUNT(DISTINCT station_id)                  AS n_stations,
           ROUND(AVG(snow_days_30cm)::numeric, 1)      AS avg_days_30cm
    FROM station_winters
    GROUP BY elevation_band, period
    ORDER BY elevation_band, period DESC
''')
q_snow

,elevation_band,period,n_stations,avg_days_30cm
0,1500 m and above,recent,9,99.5
1,1500 m and above,past,9,106.5
2,below 1500 m,recent,11,33.1
3,below 1500 m,past,11,55.3


### 4.7 Pilot resorts: which ones can be joined to the roster? (`LEFT JOIN`)

In [12]:
q_pilot = sql('''
    SELECT s.resort_id,
           s.country,
           s.risk_class,
           s.days_past,
           s.days_recent,
           CASE WHEN rr.resort_id IS NULL THEN 'no match' ELSE 'matched' END AS roster_match
    FROM snow_summary_pilot s
    LEFT JOIN resort_roster rr ON rr.resort_id = s.resort_id
    ORDER BY roster_match, s.resort_id
''')
q_pilot

,resort_id,country,risk_class,days_past,days_recent,roster_match
0,aprica,IT,reliable,123.2,113.5,matched
1,les-2-alpes,FR,reliable,128.4,132.1,matched
2,livigno,IT,reliable,135.4,132.9,matched
3,soelden,AT,reliable,136.0,135.8,matched
4,st-moritz-corviglia,CH,reliable,136.1,132.6,matched
5,andermatt-sedrun-disentis,CH,reliable,136.2,136.3,no match
6,breuil-cervinia-valtournenche-zermatt,IT,reliable,131.6,132.2,no match
7,kitzbuehel-kirchberg,AT,reliable,119.0,124.5,no match
8,les-houches-saint-gervais-chamonix,FR,reliable,124.5,135.2,no match
9,zermatt-matterhorn,CH,reliable,136.2,136.3,no match


## 5. Two queries that PostgreSQL makes simple

### 5.1 The three most expensive day passes in each country (window function)

`RANK() OVER (PARTITION BY country ORDER BY price DESC)` numbers the rows inside each country without collapsing them, as `GROUP BY` would. The outer query keeps the first three ranks. `RANK()` gives equal prices the same rank, so a country can show more than three rows when prices are tied (for example several Swiss resorts at 98.7 EUR).

In [13]:
q_top3 = sql('''
    SELECT country, price_rank, resort_name, price_day_eur
    FROM (
        SELECT country,
               resort_name,
               price_day_eur,
               RANK() OVER (PARTITION BY country ORDER BY price_day_eur DESC) AS price_rank
        FROM resorts
        WHERE price_day_eur IS NOT NULL
    ) ranked
    WHERE price_rank <= 3
    ORDER BY country, price_rank, resort_name
''')
q_top3

,country,price_rank,resort_name,price_day_eur
0,AT,1,Sölden,86.5
1,AT,2,St. Anton/​St. Christoph/​Stuben/​Lech/​Zürs/​...,85.0
2,AT,3,Ischgl/​Samnaun – Silvretta Arena,83.0
3,AT,3,Kaltenbach – Hochzillertal/​Hochfügen (SKi-opt...,83.0
4,AT,3,KitzSki – Kitzbühel/​Kirchberg,83.0
5,CH,1,Zermatt/​Breuil-Cervinia/​Valtournenche – Matt...,121.8
6,CH,2,4 Vallées – Verbier/​La Tzoumaz/​Nendaz/​Veyso...,98.7
7,CH,2,Crans-Montana,98.7
8,CH,2,Eggli/​La Videmanette – Gstaad/​Saanen/​Rougemont,98.7
9,CH,2,Rinderberg/​Saanersloch/​Horneggli – Zweisimme...,98.7


### 5.2 Median and mean price by country (`percentile_cont`)

The median is the middle price and is less affected by a few very expensive resorts than the mean.

In [14]:
q_median = sql('''
    SELECT country,
           COUNT(price_day_eur)                                                       AS n_prices,
           ROUND(AVG(price_day_eur)::numeric, 1)                                      AS mean_eur,
           ROUND((percentile_cont(0.5) WITHIN GROUP (ORDER BY price_day_eur))::numeric, 1) AS median_eur,
           MIN(price_day_eur)                                                         AS min_eur,
           MAX(price_day_eur)                                                         AS max_eur
    FROM resorts
    GROUP BY country
    ORDER BY median_eur DESC
''')
q_median

,country,n_prices,mean_eur,median_eur,min_eur,max_eur
0,AT,76,72.6,73.0,47.00,86.5
1,CH,76,73.2,70.9,40.95,121.8
2,IT,75,61.5,60.0,24.00,116.0
3,FR,73,49.7,46.0,26.00,84.2


## 6. Checks of the PostgreSQL results

The results are compared with pandas, with Matthias's published snow values and with the SQLite database from notebook 05.

In [15]:
# 1) Country averages: PostgreSQL vs pandas
check = resorts.groupby("country")["price_day_eur"].mean()
pg = q_country.set_index("country")["avg_day_pass_eur"]
assert np.allclose(check.sort_index().round(1), pg.sort_index(), atol=0.051), "Country averages differ"

# 2) No rows lost while loading
assert (load_log["lost"] == 0).all(), "Rows lost while loading"

# 3) Working hours: PostgreSQL vs pandas
latest = pd.read_csv(PROCESSED / "country_stats.csv").dropna(subset=["net_earnings_hour_eur"])
latest = latest.sort_values("year").groupby("country").tail(1).set_index("country")["net_earnings_hour_eur"]
tmp = resorts.dropna(subset=["price_day_eur"])
pandas_hours = (tmp["price_day_eur"] / tmp["country"].map(latest)).groupby(tmp["country"]).mean()
assert np.allclose(pandas_hours.sort_index(), q_hours.set_index("country")["work_hours_per_day_pass"].sort_index(), atol=0.011), "Working hours differ"

# 4) Snow: Matthias reports 55.3 -> 33.1 (11 stations) and 106.5 -> 99.5 (9 stations)
s = q_snow.set_index(["elevation_band", "period"])
assert s.loc[("below 1500 m", "past"), "avg_days_30cm"] == 55.3 and s.loc[("below 1500 m", "recent"), "avg_days_30cm"] == 33.1
assert s.loc[("1500 m and above", "past"), "avg_days_30cm"] == 106.5 and s.loc[("1500 m and above", "recent"), "avg_days_30cm"] == 99.5
assert s.loc[("below 1500 m", "past"), "n_stations"] == 11 and s.loc[("1500 m and above", "past"), "n_stations"] == 9

# 5) Duplicates: 7 groups, 14 rows
assert len(dups) == 7 and dups["n_rows"].sum() == 14

# 6) Same row counts as in the SQLite database
if SQLITE_PATH.exists():
    lite = sqlite3.connect(SQLITE_PATH)
    for table in load_log["table"]:
        n_lite = lite.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        n_pg = int(sql(f"SELECT COUNT(*) AS n FROM {table}")["n"].iloc[0])
        assert n_lite == n_pg, f"{table}: SQLite {n_lite} rows, PostgreSQL {n_pg} rows"
    # the median of Italy must also be the same in pandas
    assert abs(float(q_median.set_index("country").loc["IT", "median_eur"]) - round(float(resorts.loc[resorts.country == "IT", "price_day_eur"].median()), 1)) < 0.051
    lite.close()
    print("SQLite and PostgreSQL contain the same number of rows in every table.")

# 7) Top 3 per country: exactly 3 rows per country unless there are ties
assert q_top3.groupby("country").size().min() >= 3

print("All checks passed.")

SQLite and PostgreSQL contain the same number of rows in every table.
All checks passed.


## 7. Integration summary (copied into `docs/integration_log.md`)

In [16]:
load_log

,table,source,rows_in_file,rows_in_table,lost
0,resort_roster,resorts_snow_roster.csv,293,293,0
1,resorts,resorts_features.csv,300,300,0
2,closures,legambiente_closures.csv,184,184,0
3,country_stats,country_stats.csv,123,123,0
4,fx_eur_chf_daily,fx_eur_chf_daily.csv,4098,4098,0
5,station_winters,meteoswiss_station_winters.csv,400,400,0
6,snow_summary_pilot,snow_summary_10resorts.csv,10,10,0
7,snow_winters_pilot,snow_winters_pilot.csv,20,20,0


In [17]:
engine.dispose()
print("PostgreSQL database", PG_DB, "on", PG_HOST + ":" + PG_PORT, "- notebook finished")

PostgreSQL database sci_alpi on localhost:5432 - notebook finished
